# 04 — Generation Demo

Load a trained checkpoint and generate text from custom prompts.
Compare outputs across different temperatures and top-k values.

In [1]:
import sys
sys.path.insert(0, '..')

import torch
import tiktoken

from src.infra.config import load_config
from src.infra.io import load_checkpoint
from src.models.gpt.model import GPT
from src.core.generation import generate

In [2]:
# ── Load config and checkpoint ────────────────────────────────────────────
config = load_config('../configs/experiments/exp_001_baseline.yaml')
device = 'cuda' if torch.cuda.is_available() else 'cpu'

model = GPT(config.model).to(device)
ckpt_path = '../outputs/checkpoints/best_model.pt'
load_checkpoint(ckpt_path, model, device=device)
model.eval()

enc = tiktoken.get_encoding('gpt2')
print(f'Model loaded on {device}')

Model loaded on cpu


In [3]:
# ── Generate text ─────────────────────────────────────────────────────────
def generate_text(prompt: str, max_tokens: int = 200,
                  temperature: float = 1.0, top_k: int = 50) -> str:
    tokens = enc.encode_ordinary(prompt)
    ctx = torch.tensor(tokens, dtype=torch.long, device=device).unsqueeze(0)
    out = generate(model, ctx, max_new_tokens=max_tokens,
                   temperature=temperature, top_k=top_k)
    return enc.decode(out.squeeze().tolist())

prompts = [
    'Once upon a time there was a pumpkin.',
    'A little girl went to the woods',
]
for p in prompts:
    print(f'--- Prompt: "{p}" ---')
    print(generate_text(p, max_tokens=200))
    print()

--- Prompt: "Once upon a time there was a pumpkin." ---
Once upon a time there was a pumpkin. It was a magical shell and shiny. One day, they were on a adventure. When it was time to go home, Anna and her brother, had a big, scary surprise for her to eat. Anna ran to her house to buy a shell to sleep.

"What's a beautiful, Anna?" Mom asked. "It's a nice boat!"

Anna smiled and wanted to help her brother's for her. She did not believe how much she was. She took another carrot and put it on. She rubbed it in her nose and said, "I'm a big heart for you! I know you are silly!"

Anna smiled. She said, "Sure, Anna. I will help you."

Mom looked at Anna and sighed. She said, "Yes, thank you for playing. Then we can make a tent together. Please let's go!"

She made cookies in the room and opened them. She smiled. She said, "No

--- Prompt: "A little girl went to the woods" ---
A little girl went to the woods. She saw a bird with lots of feathers and legs. She was very curious. She was not scar

In [4]:
# ── Temperature sweep ─────────────────────────────────────────────────────
prompt = 'Once upon a time'
for temp in [0.5, 0.8, 1.0, 1.2]:
    print(f'--- temperature={temp} ---')
    print(generate_text(prompt, max_tokens=80, temperature=temp, top_k=50))
    print()

--- temperature=0.5 ---
Once upon a time, there was a little girl named Lily. She loved to play outside in the sunshine. One day, she saw a big tree with lots of leaves. She wanted to climb it, but her mom said no.

Lily went to the tree and saw a big tree. She was scared and didn't know what to do. But then, she saw a big tree. She tried to

--- temperature=0.8 ---
Once upon a time, there was a bald man who lived in a big house. He had a big house. He liked to play with his boat.

One day, the man came to the office and saw the bull. He was scared and didn't know how to play. He wanted to take a nap.

"Don't be angry!" the man said. "Don't be scared, little

--- temperature=1.0 ---
Once upon a time, there was a big white balloon that lived in a beautiful sunshine. It was very excited, so it didn't see it. It rolled over and around but it was too dark. Every day the balloon would go outside and spin around the yard. But when they got there, it had no other people.

So the bird thought 